# CeView forecasting: Colab training and results

Run in a **Google Colab GPU runtime**. This notebook imports repository code to run the three baselines and one selected experiment per session, log results and best weights to Weights & Biases, and show training, validation, and the selected model's final test report.

Before running, place this project in your Google Drive at MyDrive/Ceview_Transformer. Include the ignored dataset, raw CSVs, and archived exporter listed in the frozen protocol, not just the Git checkout. Add your W&B key to the project .env; never paste it into a notebook cell.

The current dataset is provisional. Preflight will stop until actual collection verification, diagnostic approval, and data-owner permissions are recorded in a reviewed protocol revision. Do not set approval flags merely to bypass the check.

Set EXPERIMENT to "A" for the first session, "B" tomorrow, and "C" when ready. Each selection runs its three frozen seeds only. Use the same project, protocol, output folder, and compatible runtime across sessions.

Once setup and dataset review are complete, use **Runtime > Run all**. Drive may request access. Completed runs are reused; an interrupted training run starts a new same-seed attempt. A completed test report is displayed again without reevaluating the test set.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Experiment selection and locations
Change these paths if your Drive folders use different names. Keep OUTPUT_ROOT unchanged across reruns so the test ledger persists.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

EXPERIMENT = "A"  # Choose A today, B tomorrow, or C in a later session.

PROJECT_ROOT = Path("/content/drive/MyDrive/Ceview_Transformer")
OUTPUT_ROOT = Path("/content/drive/MyDrive/CeviewTraining")
PROTOCOL_PATH = PROJECT_ROOT / "experiments/forecasting-v2-wandb.json"

if not (PROJECT_ROOT / "training/requirements.txt").is_file():
    raise FileNotFoundError("Copy the project into PROJECT_ROOT before continuing.")
if not PROTOCOL_PATH.is_file():
    raise FileNotFoundError("The selected frozen protocol is missing.")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

## Dependencies
Install before importing the model packages. If preflight detects an already-loaded incompatible package, restart the runtime and run all cells again.

In [ ]:
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(PROJECT_ROOT / "training/requirements.txt")
])

## Configuration and preflight
Only non-secret configuration is displayed. .env is loaded by the logging helper and is never included in W&B artifacts.

In [ ]:
import torch
import pandas as pd
from IPython.display import display
from model.experiment import load_protocol
from training.tracking import configure_wandb
from training.workflow import preflight, run_suite, display_results

protocol = load_protocol(PROTOCOL_PATH)
display(pd.DataFrame([{
    "protocol": protocol["protocol_version"],
    "dataset": protocol["dataset"]["version"],
    "status": protocol["dataset"]["status"],
    "diagnostic_approval": protocol["dataset"]["diagnostic_approval"],
    "training_examples": protocol["dataset"]["counts"]["train"],
    "validation_examples": protocol["dataset"]["counts"]["validation"],
    "test_examples": protocol["dataset"]["counts"]["test"],
    "GPU_available": torch.cuda.is_available(),
}]))
device = torch.device("cuda")
preflight(PROJECT_ROOT, PROTOCOL_PATH, device)
configure_wandb(PROJECT_ROOT / ".env")
print("Preflight passed; credentials loaded without displaying the key.")

## Run the selected experiment

Epoch MAE values appear below and stream to W&B. The runner saves each best validation checkpoint to Drive and uploads it as a W&B artifact on run completion. It saves the selected experiment's training and validation results, then stops if other experiments are incomplete. Once all nine runs across A/B/C exist, it records validation-only selection before opening the test partition. If no architecture beats the strongest baseline, it reports that result and leaves test data sealed.

In [ ]:
suite = run_suite(PROJECT_ROOT, PROTOCOL_PATH, OUTPUT_ROOT, device=device, experiment=EXPERIMENT)

## Results
Show all completed training curves, per-seed scores, detailed validation results, baselines, and W&B links, including partial A-only or A+B progress. Architecture comparison and final test scores appear only after all three experiments are complete.

In [ ]:
display_results(suite)

## Saved outputs

Drive contains the suite record, baseline metrics, per-run epoch CSVs and best weights, selection record, and final test JSON. W&B contains corresponding runs, tables, and explicitly selected model/evaluation artifacts. Dataset files, .env, and source folders are not uploaded as artifacts.

The persistent test-evaluations folder guards against automatic repeated test evaluation. If a runtime fails after claiming the test but before saving the report, the runner stops for a ledger review instead of silently reevaluating. Do not delete the ledger to compare another candidate against test results.